# Módulo X — Agentes Inteligentes

Código do livro *Ciência de Dados com Python* (Howard Roatti). Explicações no livro; repositório: https://github.com/howardroatti/roatti-ciencia-de-dados-python

In [ ]:
import ast, json, operator, re, tempfile, os
from dataclasses import dataclass
from typing import Callable
from pydantic import BaseModel, Field, ValidationError

import sys
sys.path.insert(0, "..")          # permite `import src...` a partir de codigo/notebooks/
from src.dados_sinteticos_m10_m11 import FATOS_AGENTE

## Uso de Ferramentas (*Tool Use / Function Calling*)

In [ ]:
ferramentas = [{
    "type": "function",
    "function": {
        "name": "consultar_clima",
        "description": "Retorna a temperatura atual de uma cidade.",
        "parameters": {
            "type": "object",
            "properties": {
                "cidade": {"type": "string", "description": "Nome da cidade"}
            },
            "required": ["cidade"],
        },
    },
}]

In [ ]:
# OPCIONAL: requer `pip install ollama`, servidor Ollama ativo e `ollama pull llama3.2:3b`
import ollama

def consultar_clima(cidade: str) -> str:
    # numa aplicação real, aqui iria uma chamada a uma API meteorológica
    return f"A temperatura em {cidade} é 28°C."

mensagens = [{"role": "user", "content": "Qual a temperatura em Recife?"}]
resposta = ollama.chat(model="llama3.2:3b", messages=mensagens, tools=ferramentas)

# o modelo decidiu chamar uma ferramenta?
for chamada in resposta["message"].get("tool_calls", []):
    nome = chamada["function"]["name"]
    args = chamada["function"]["arguments"]
    if nome == "consultar_clima":
        resultado = consultar_clima(**args)                       # NOSSO código executa
        mensagens.append(resposta["message"])                     # registra a decisão do modelo
        mensagens.append({"role": "tool", "content": resultado})  # devolve a observação

# segunda passada: agora o modelo redige a resposta final com o dado real
final = ollama.chat(model="llama3.2:3b", messages=mensagens, tools=ferramentas)
print(final["message"]["content"])

## O Padrão ReAct: Raciocinar + Agir

In [ ]:
# OPCIONAL: requer `pip install ollama`, servidor Ollama ativo e `ollama pull llama3.2:3b`
import ollama

# --- ferramentas do agente ---
def calculadora(expressao: str) -> str:
    try:
        return str(eval(expressao, {"__builtins__": {}}))  # sandbox mínimo (ver Módulo XI)
    except Exception as e:
        return f"erro: {e}"

def buscar_base(consulta: str) -> str:
    # aqui entraria um RAG (Módulo IX); simplificado para o exemplo
    fatos = {"capital do brasil": "Brasília", "populacao de brasilia": "cerca de 2,8 milhões"}
    return fatos.get(consulta.lower(), "sem resultado")

TOOLS = {"calculadora": calculadora, "buscar_base": buscar_base}

esquema = [
    {"type": "function", "function": {"name": "calculadora",
        "description": "Avalia uma expressão matemática. Ex: '2*(3+4)'.",
        "parameters": {"type": "object",
            "properties": {"expressao": {"type": "string"}}, "required": ["expressao"]}}},
    {"type": "function", "function": {"name": "buscar_base",
        "description": "Busca um fato numa base de conhecimento interna.",
        "parameters": {"type": "object",
            "properties": {"consulta": {"type": "string"}}, "required": ["consulta"]}}},
]

def agente(objetivo: str, max_passos: int = 5) -> str:
    msgs = [
        {"role": "system", "content": "Você é um agente. Use ferramentas quando precisar. "
                                      "Quando tiver a resposta final, responda diretamente."},
        {"role": "user", "content": objetivo},
    ]
    for passo in range(max_passos):                 # laço = orquestração; limite evita loop infinito
        r = ollama.chat(model="llama3.2:3b", messages=msgs, tools=esquema)
        chamadas = r["message"].get("tool_calls")
        if not chamadas:                            # sem ação => resposta final (condição de parada)
            return r["message"]["content"]
        msgs.append(r["message"])
        for c in chamadas:                          # AÇÃO + OBSERVAÇÃO
            nome, args = c["function"]["name"], c["function"]["arguments"]
            observacao = TOOLS[nome](**args)
            print(f"[passo {passo}] ação={nome}{args} -> obs={observacao}")
            msgs.append({"role": "tool", "content": observacao})
    return "Limite de passos atingido."

print(agente("Quanto é a população de Brasília multiplicada por 2? Responda o número final."))

## Mão na Massa: um Agente do Zero (sem LLM externo)

### Um registro de ferramentas

In [ ]:
@dataclass
class Ferramenta:
    nome: str
    descricao: str
    args_modelo: type[BaseModel]      # Pydantic valida e gera o esquema JSON
    funcao: Callable

    def esquema(self) -> dict:        # mesmo formato "function" usado no início do módulo
        return {"type": "function", "function": {
            "name": self.nome, "description": self.descricao,
            "parameters": self.args_modelo.model_json_schema()}}

REGISTRO: dict[str, Ferramenta] = {}

def ferramenta(nome: str, descricao: str, args_modelo: type[BaseModel]):
    def decorador(funcao):
        REGISTRO[nome] = Ferramenta(nome, descricao, args_modelo, funcao)
        return funcao
    return decorador

In [ ]:
class ArgsCalc(BaseModel):
    expressao: str = Field(description="Expressão aritmética, ex.: '2*(3+4)'")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def _avaliar(no):
    if isinstance(no, ast.Expression):
        return _avaliar(no.body)
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in _OPS:
        return _OPS[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and type(no.op) in _OPS:
        return _OPS[type(no.op)](_avaliar(no.operand))
    raise ValueError("expressão não permitida")

@ferramenta("calculadora", "Avalia uma expressão aritmética.", ArgsCalc)
def calculadora(expressao: str) -> str:
    return str(_avaliar(ast.parse(expressao, mode="eval")))

class ArgsBusca(BaseModel):
    consulta: str = Field(description="Chave do fato, ex.: 'capital do brasil'")

@ferramenta("buscar_base", "Busca um fato na base de conhecimento interna.", ArgsBusca)
def buscar_base(consulta: str) -> str:
    return FATOS_AGENTE.get(consulta.lower().strip(), "sem resultado")

print(json.dumps(REGISTRO["calculadora"].esquema(), ensure_ascii=False, indent=1))

### Memória de longo prazo, na versão mínima

In [ ]:
class MemoriaLongoPrazo:
    """Memória persistente chave-valor em JSON (versão mínima do 'banco de memórias')."""
    def __init__(self, caminho: str):
        self.caminho = caminho
        self.dados = json.load(open(caminho, encoding="utf-8")) if os.path.exists(caminho) else {}

    def guardar(self, chave: str, valor: str) -> None:
        self.dados[chave.lower()] = valor
        with open(self.caminho, "w", encoding="utf-8") as f:
            json.dump(self.dados, f, ensure_ascii=False, indent=1)

    def recuperar(self, chave: str) -> str | None:
        return self.dados.get(chave.lower())

CAMINHO_MEMORIA = os.path.join(tempfile.gettempdir(), "memoria_agente_demo.json")
if os.path.exists(CAMINHO_MEMORIA):
    os.remove(CAMINHO_MEMORIA)                # começa a demonstração do zero
MEMORIA = MemoriaLongoPrazo(CAMINHO_MEMORIA)

class ArgsGuardar(BaseModel):
    chave: str
    valor: str

class ArgsRecordar(BaseModel):
    chave: str

@ferramenta("guardar_memoria", "Guarda um fato na memória de longo prazo.", ArgsGuardar)
def guardar_memoria(chave: str, valor: str) -> str:
    MEMORIA.guardar(chave, valor)
    return f"guardado: {chave}"

@ferramenta("recordar_memoria", "Recupera um fato da memória de longo prazo.", ArgsRecordar)
def recordar_memoria(chave: str) -> str:
    return MEMORIA.recuperar(chave) or "nada guardado com essa chave"

### O LLM simulado

In [ ]:
def llm_simulado(mensagens: list[dict]) -> str:
    """Política por regras que imita um LLM no formato ReAct (determinística)."""
    objetivo = next(m["content"] for m in mensagens if m["role"] == "user").lower()
    obs = [m["content"] for m in mensagens if m["role"] == "tool"]
    n = len(obs)

    def agir(pensamento, nome, args):
        return f"Thought: {pensamento}\nAction: {nome}\nAction Input: {json.dumps(args, ensure_ascii=False)}"

    if "população de brasília" in objetivo and "2" in objetivo:      # busca -> cálculo -> resposta
        if n == 0:
            return agir("Preciso do dado da população.", "buscar_base", {"consulta": "populacao de brasilia"})
        if n == 1:
            return agir(f"Tenho {obs[0]}; falta multiplicar por 2.", "calculadora", {"expressao": f"{obs[0]}*2"})
        return f"Thought: Já tenho o resultado.\nFinal Answer: {int(float(obs[1])):,}".replace(",", ".")
    if objetivo.startswith("lembre que"):                            # memória de longo prazo
        chave, valor = objetivo.removeprefix("lembre que ").split(" é ")
        if n == 0:
            return agir("Vou guardar esse fato.", "guardar_memoria", {"chave": chave, "valor": valor})
        return "Thought: Guardado.\nFinal Answer: Anotado."
    if objetivo.startswith("qual é "):
        chave = objetivo.removeprefix("qual é ").rstrip("?")
        if n == 0:
            return agir("Vou consultar a memória.", "recordar_memoria", {"chave": chave})
        return f"Thought: Recuperei o fato.\nFinal Answer: {obs[0]}"
    if "enviar e-mail" in objetivo:                                  # ferramenta inexistente
        if n == 0:
            return agir("Vou enviar o e-mail.", "enviar_email", {"para": "x@exemplo.com"})
        return f"Thought: A ferramenta falhou.\nFinal Answer: Não consegui: {obs[-1]}"
    if "raiz de" in objetivo:                                        # argumento inválido
        if n == 0:
            return agir("Vou calcular.", "calculadora", {"expr": "raiz(9)"})
        return f"Thought: Argumento recusado.\nFinal Answer: Não consegui: {obs[-1]}"
    if "dividir por zero" in objetivo:                               # erro na própria ferramenta
        if n == 0:
            return agir("Vou dividir.", "calculadora", {"expressao": "1/0"})
        return f"Thought: Houve erro.\nFinal Answer: Não consegui: {obs[-1]}"
    if "teimoso" in objetivo:                                        # nunca termina
        return agir("Vou tentar de novo.", "buscar_base", {"consulta": "algo que não existe"})
    return "Thought: Posso responder direto.\nFinal Answer: Não sei fazer isso com as ferramentas que tenho."

### O laço ReAct completo

In [ ]:
PADRAO_ACAO = re.compile(r"Action:\s*(\w+)\s*\nAction Input:\s*(\{.*\})", re.S)
PADRAO_FINAL = re.compile(r"Final Answer:\s*(.*)", re.S)

def executar_ferramenta(nome: str, args: dict) -> str:
    """Executa com validação; devolve SEMPRE texto (erros viram observações)."""
    if nome not in REGISTRO:
        return f"erro: ferramenta '{nome}' não existe. Disponíveis: {sorted(REGISTRO)}"
    ferr = REGISTRO[nome]
    try:
        validos = ferr.args_modelo(**args)             # valida tipos e campos obrigatórios
    except ValidationError as e:
        campos = ", ".join(str(err["loc"][0]) for err in e.errors())
        return f"erro: argumentos inválidos ({campos})"
    try:
        return ferr.funcao(**validos.model_dump())
    except Exception as e:
        return f"erro na ferramenta: {type(e).__name__}: {e}"

def agente_react(objetivo: str, llm=llm_simulado, max_passos: int = 5, verbose: bool = True):
    historico = [{"role": "system", "content": "Responda no formato Thought/Action/Final Answer."},
                 {"role": "user", "content": objetivo}]                 # memória de curto prazo
    for passo in range(1, max_passos + 1):
        saida = llm(historico)
        historico.append({"role": "assistant", "content": saida})
        final = PADRAO_FINAL.search(saida)
        if final:                                                      # condição de parada
            if verbose: print(f"[{passo}] Final Answer: {final.group(1).strip()}")
            return final.group(1).strip(), historico
        acao = PADRAO_ACAO.search(saida)
        if not acao:
            obs = "erro: formato inválido; use Action/Action Input ou Final Answer"
        else:
            try:
                args = json.loads(acao.group(2))
                obs = executar_ferramenta(acao.group(1), args)
            except json.JSONDecodeError:
                obs = "erro: Action Input não é JSON válido"
        if verbose: print(f"[{passo}] ação={acao.group(1) if acao else '?'} -> obs={obs}")
        historico.append({"role": "tool", "content": obs})              # OBSERVAÇÃO
    return "Limite de passos atingido.", historico

In [ ]:
resposta, hist = agente_react("Quanto é a população de Brasília multiplicada por 2?")
print("Resposta:", resposta)
print()
for m in hist[1:]:
    print(f"{m['role']:>9}: {m['content']}")

### Quando as coisas dão errado

In [ ]:
for objetivo in ["Preciso enviar e-mail ao cliente.",
                 "Calcule a raiz de 9.",
                 "Tente dividir por zero.",
                 "Seja teimoso."]:
    print("OBJETIVO:", objetivo)
    r, _ = agente_react(objetivo, max_passos=3)
    print()

### Memória em ação

In [ ]:
agente_react("Lembre que desconto máximo é 15%")
agente_react("Qual é desconto máximo?")
nova_sessao = MemoriaLongoPrazo(CAMINHO_MEMORIA)      # simula reabrir o programa
print(nova_sessao.dados)

### Testando o agente

In [ ]:
def test_sucesso():
    r, h = agente_react("Quanto é a população de Brasília multiplicada por 2?", verbose=False)
    assert r == "5.600.000"
    assert [m["role"] for m in h].count("tool") == 2

def test_ferramenta_inexistente():
    assert "não existe" in executar_ferramenta("enviar_email", {})

def test_argumento_invalido():
    assert "argumentos inválidos" in executar_ferramenta("calculadora", {"expr": "1+1"})
    assert "argumentos inválidos" in executar_ferramenta("calculadora", {"expressao": 123})

def test_calculadora_bloqueia_codigo():
    assert "erro na ferramenta" in executar_ferramenta("calculadora", {"expressao": "__import__('os')"})

def test_limite_de_passos():
    r, h = agente_react("Seja teimoso.", max_passos=4, verbose=False)
    assert r == "Limite de passos atingido."
    assert [m["role"] for m in h].count("tool") == 4

def test_formato_invalido():
    r, _ = agente_react("qualquer", llm=lambda msgs: "bla bla", max_passos=2, verbose=False)
    assert r == "Limite de passos atingido."

for nome, fn in list(globals().items()):
    if nome.startswith("test_"):
        fn()
        print("ok  ", nome)

### Trocando o LLM simulado pelo real

In [ ]:
# OPCIONAL: requer `pip install ollama`, o servidor Ollama em execução e `ollama pull llama3.2:3b`
import ollama

SISTEMA_REACT = """Você é um agente. Ferramentas disponíveis:
{ferramentas}
Responda SEMPRE em um destes formatos:
Thought: <raciocínio>
Action: <nome da ferramenta>
Action Input: <JSON com os argumentos>
ou
Thought: <raciocínio>
Final Answer: <resposta>"""

def llm_ollama(mensagens: list[dict]) -> str:
    descricao = "\n".join(f"- {f.nome}: {f.descricao} {list(f.args_modelo.model_fields)}"
                          for f in REGISTRO.values())
    msgs = [{"role": "system", "content": SISTEMA_REACT.format(ferramentas=descricao)}]
    for m in mensagens[1:]:                          # troca o system prompt simples pelo completo
        papel = "user" if m["role"] == "tool" else m["role"]
        conteudo = f"Observation: {m['content']}" if m["role"] == "tool" else m["content"]
        msgs.append({"role": papel, "content": conteudo})
    return ollama.chat(model="llama3.2:3b", messages=msgs,
                       options={"temperature": 0, "stop": ["Observation:"]})["message"]["content"]

# resposta, _ = agente_react("Quanto é a população de Brasília multiplicada por 2?", llm=llm_ollama)